# Notebook 05 — RAG Weather Retrieval
**Syllabus Week 10 | CLO 5: Information Retrieval & Embeddings**

## Problem Statement
Build a **Retrieval-Augmented Generation** pipeline for Pakistani weather knowledge.

Given a natural-language query (e.g. "Is Lahore monsoon usually this humid?"),
retrieve the top-k most relevant snippets from a 330-entry curated corpus and
return them as grounding context to the LLM agent.

**Components:**
- Encoder: `sentence-transformers/all-MiniLM-L6-v2` (22M params, 384-dim)
- Index: FAISS `IndexFlatIP` (cosine similarity via L2-normalised inner product)
- Corpus: `ml/datasets/weather_snippets.json` (330 snippets, 6 cities × 5 seasons × 5 event types)

| Parameter | Value |
|---|---|
| Embedding dim | 384 |
| Index type | FAISS IndexFlatIP |
| Similarity | Cosine (via L2-normalised IP) |
| Corpus size | 330 snippets |
| Default k | 3 |
| Build time | < 5 s on CPU |

## Section 2 — Math Derivation

### Sentence Embedding
MiniLM encodes text $t$ to a dense vector via mean-pooling over token embeddings
and L2-normalisation:

$$e = \text{Encoder}(t), \quad \hat{e} = \frac{e}{\|e\|_2}$$

### Cosine Similarity via Inner Product
After L2-normalisation, inner product equals cosine similarity:

$$\text{cos}(\hat{q}, \hat{d}) = \hat{q} \cdot \hat{d} = \sum_{i} \hat{q}_i \hat{d}_i \in [-1, 1]$$

FAISS `IndexFlatIP` computes exact brute-force inner product over all $N$ corpus
vectors — no approximation, correct for $N \le 10^5$.

### Top-k Retrieval
$$\mathcal{R}_k(q) = \underset{d \in \mathcal{D}}{\text{top-}k} \; \hat{q} \cdot \hat{d}$$

Returned with scores so the LLM can weight low-scoring results differently.

### Why Not BM25?
BM25 is lexical — it misses semantic synonyms ("hot" ≠ "scorching" in BM25 but
they map to nearby regions in embedding space). Dense retrieval with MiniLM
generalises across paraphrases, which is critical for weather queries that
use colloquial Pakistani language.

### RAG Augmentation Formula
Given retrieved documents $\mathcal{R}_k(q) = \{d_1, \ldots, d_k\}$, the LLM
is conditioned on the concatenated context:
$$\hat{y} = P_{\text{LLM}}(y \mid q, d_1, \ldots, d_k)$$

This reduces hallucination about city-specific weather facts without requiring
fine-tuning the base LLM.

In [ ]:
# Section 3 — Corpus Loading
import sys; sys.path.insert(0, '..')
import json
from pathlib import Path

# corpus_path = Path('../datasets/weather_snippets.json')
# with open(corpus_path, encoding='utf-8') as f:
#     snippets = json.load(f)
# print(f'Total snippets: {len(snippets)}')
# print(f'Example: {snippets[0]}')

# Breakdown by generated vs hand-curated:
# generated   = [s for s in snippets if s.get('generated')]
# hand_curated = [s for s in snippets if not s.get('generated')]
# print(f'Hand-curated: {len(hand_curated)}')
# print(f'Generated   : {len(generated)}')

# Tag distribution:
# from collections import Counter
# all_tags = [t for s in snippets for t in s.get('tags', [])]
# print(Counter(all_tags).most_common(15))

In [ ]:
# Section 4 — Index Build & Embedding
# TODO: run after pip install sentence-transformers faiss-cpu

# from ml.infer.rag_infer import build_index, retrieve_weather_context
# import asyncio

# Build index (encodes all 330 snippets with MiniLM, ~3s on CPU)
# build_index()   # idempotent — rebuilds from JSON each call

# Embedding shapes:
# texts    : List[str]   len=330
# embeddings : np.ndarray  (330, 384)   float32, L2-normalised
# index    : faiss.IndexFlatIP  ntotal=330  d=384

# Sample queries to verify routing:
# queries = [
#     'Is Lahore monsoon usually this humid?',
#     'Why is Quetta so cold in winter?',
#     'Karachi heat wave during pre-monsoon',
# ]
# for q in queries:
#     results = asyncio.run(retrieve_weather_context(q, k=3))
#     print(f'\nQ: {q}')
#     for r in results:
#         print(f'  [{r["score"]:.3f}] {r["text"][:80]}...')

In [ ]:
# Section 5 — Build Time & Query Latency Benchmark
import time

# from ml.infer.rag_infer import build_index, retrieve_weather_context
# import asyncio

# Build time
# t0 = time.perf_counter()
# build_index()
# build_time = time.perf_counter() - t0
# print(f'Index build time: {build_time:.2f}s')   # target < 5s

# Query latency (100 queries, average)
# test_q = 'temperature extremes in Peshawar summer'
# t0 = time.perf_counter()
# for _ in range(100):
#     asyncio.run(retrieve_weather_context(test_q, k=3))
# avg_ms = (time.perf_counter() - t0) / 100 * 1000
# print(f'Average query latency: {avg_ms:.1f} ms')   # target < 50ms

# TODO: paste results here
# Index build time: X.XXs
# Average query latency: X.Xms

## Section 6 — Architecture Diagram

```
OFFLINE (build_index, called once at server startup)
──────────────────────────────────────────────────
weather_snippets.json (330 texts)
      │
 MiniLM Encoder                    22M params, 384-dim
      │  encode(texts, batch=64)
 embeddings  (330, 384)  float32
      │  L2-normalise each row
 FAISS IndexFlatIP.add(embeddings)
      │
 [index in RAM]


ONLINE (retrieve_weather_context, called per agent turn)
────────────────────────────────────────────────────────
query string
      │
 MiniLM Encoder                    same weights, reused
      │  (1, 384)
 L2-normalise
      │
 FAISS IndexFlatIP.search(q, k=3)  exact brute-force
      │  scores (1, 3)  indices (1, 3)
 look up snippets[idx]
      │
 [{text, score, tags}, ...]         returned to agent
```

In [ ]:
# Section 7 — Qualitative Evaluation
import matplotlib.pyplot as plt
import numpy as np

# 7a. Similarity score distribution across 330 snippets for a fixed query
# from ml.infer.rag_infer import _get_index  # internal helper
# from sentence_transformers import SentenceTransformer
# encoder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
# q = encoder.encode(['monsoon flooding in Pakistan'], normalize_embeddings=True)
# index, texts = _get_index()
# scores, _ = index.search(q, len(texts))
# plt.figure(figsize=(8, 3))
# plt.hist(scores[0], bins=40, edgecolor='k')
# plt.xlabel('Cosine similarity')
# plt.ylabel('Count')
# plt.title('Score distribution — "monsoon flooding" query over 330 snippets')
# plt.tight_layout(); plt.show()

# 7b. Precision@k: verify top-3 are monsoon / rainfall snippets
# results = asyncio.run(retrieve_weather_context('monsoon flooding Islamabad', k=3))
# for r in results:
#     print(f"[{r['score']:.3f}] tags={r['tags']}")
#     print(f"  {r['text'][:120]}...")
#     print()

# TODO: paste top-3 results for 3 diverse queries to demonstrate retrieval quality

## Section 8 — Reflection & Trade-offs

### Results (fill in after running)
| Metric | Value |
|---|---|
| Index build time | TODO s |
| Query latency (avg) | TODO ms |
| Corpus size | 330 snippets |
| Embedding dim | 384 |
| Index RAM footprint | ~330×384×4 bytes ≈ 0.5 MB |

### Design Decisions
| Decision | Alternative | Reason |
|---|---|---|
| FAISS IndexFlatIP | HNSW / IVF | Exact search is correct for N=330; approximate needed only for N>100K |
| MiniLM-L6-v2 | all-mpnet-base-v2 (768-dim) | 384-dim is 2× faster with comparable accuracy for short passages |
| L2-normalise + IP | IndexFlatL2 | Cosine is better than Euclidean for semantic similarity |
| Build at startup | Build offline + persist | 330 snippets encode in 3s — not worth maintaining a saved index file |
| Generated corpus (300) + hand-curated (30) | All hand-curated | Combinatorial coverage impossible by hand; generated text is varied enough |

### Limitations
- No re-ranking step (e.g. cross-encoder) — first-stage retrieval may surface
  plausible but slightly off-topic snippets.
- Corpus is synthetic / template-generated — lacks real incident reports.
- No hybrid BM25+dense retrieval — exact keyword matches (e.g. city names) rely
  on the embedding model seeing them in training data.

### Numbers to fill in
- Highest similarity score on best query: **TODO**
- Lowest similarity score in top-3 (worst query): **TODO**
- Any misrouted retrieval observed: **TODO**